# 00 - Setup check
Checks that (1) the Colab T4 GPU is attached, (2) Hugging Face login works, (3) FLORES-200 loads and NLLB-600M translates.

Run the cells in order. This notebook runs on Google's GPU, not your laptop.

In [ ]:
# Cell 1: is the GPU attached?
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
# Cell 2: log in to Hugging Face (needed for the gated FLORES+ dataset)
# Tries the Colab secret HF_TOKEN first; if that is not available, asks you to paste the token.
from huggingface_hub import login
token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception as e:
    print("Colab secret not available here:", type(e).__name__)
if not token:
    from getpass import getpass
    token = getpass("Paste your Hugging Face token (it will not be shown): ")
login(token=token)
print("Logged in to Hugging Face.")

In [ ]:
# Cell 3: load FLORES-200 and translate 5 sentences English -> Hindi with NLLB-600M
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

eng = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest")
hin = load_dataset("openlanguagedata/flores_plus", "hin_Deva", split="devtest")
print(len(eng), "sentences")  # should be 1012

name = "facebook/nllb-200-distilled-600M"
tok = AutoTokenizer.from_pretrained(name, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(name, torch_dtype=torch.float16).cuda()

src = eng["text"][:5]
batch = tok(src, return_tensors="pt", padding=True).to("cuda")
out = model.generate(**batch,
                     forced_bos_token_id=tok.convert_tokens_to_ids("hin_Deva"),
                     num_beams=4, max_length=256)
preds = tok.batch_decode(out, skip_special_tokens=True)

for s, p, r in zip(src, preds, hin["text"][:5]):
    print("EN  :", s)
    print("NLLB:", p)
    print("REF :", r)
    print()